# Transfection results

Reads `analysis/` from `analyze.ipynb` and the samples already stored in `assay.json`. Tables (`*.xlsx`) and Plots (`*.png`) are separate cells, so figures can be regenerated without rewriting spreadsheets.

This notebook does not name samples and does not change positions. Edit those on Studio Metadata, then re-run from crop if the position list changed.


## Config


In [ ]:
from pathlib import Path

from lisca.core.assay_positions import load_assay_positions
from transfection.core import SampleAnalysis, load_assay_for_workspace

# Folder with analysis/ and assay.json from Studio.
WORKSPACE = Path(r"Z:\projects\LNPbinder\Experiments\20260731\Auswertung")

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")

positions = load_assay_positions(workspace)
config = load_assay_for_workspace(workspace)
if config.assay_type != "transfection":
    raise ValueError(
        f"assay.json type is {config.assay_type!r}. "
        "These notebooks group transfection samples. Open this assay in Studio."
    )
if not config.mapping:
    raise ValueError("assay.json has no samples. Name each sample on Studio Metadata.")

mapping = {}
for name, entry in config.mapping.items():
    file_ids = positions.samples.get(name)
    if not file_ids:
        raise ValueError(f"Sample {name!r} has no positions in assay.json.")
    mapping[name] = SampleAnalysis(
        name=entry.name,
        positions=file_ids,
        signal_channels=list(entry.signal_channels),
        segmentation_channel=entry.segmentation_channel,
    )
interval_minutes = config.interval_minutes
print(f"Workspace: {workspace}")
print(f"Samples from assay.json: {', '.join(mapping)}")
print(f"Interval: {interval_minutes} minutes")


## Tables


In [ ]:
from transfection.core import publish_sample_tables_xlsx, publish_sample_traces_xlsx

for path in publish_sample_traces_xlsx(workspace, mapping):
    print(f"Wrote table: {path}")
for path in publish_sample_tables_xlsx(workspace, mapping, "auc"):
    print(f"Wrote table: {path}")
for path in publish_sample_tables_xlsx(workspace, mapping, "fit"):
    print(f"Wrote table: {path}")


## Plots

Re-run Plots without re-running Tables to regenerate figures.


In [ ]:
from transfection.services import plot_auc, plot_fit, plot_traces

for path in plot_traces.run_plot_traces(
    metrics_dir=workspace,
    interval=interval_minutes,
):
    print(plot_traces.format_written_traces_plot_message(path))

for message in plot_auc.format_written_auc_plot_messages(
    list(plot_auc.run_plot_auc(auc_csv=workspace))
):
    print(message)

for message in plot_fit.format_written_fit_plot_messages(
    plot_fit.run_plot_fit(
        workspace,
        output=None,
        interval=interval_minutes,
        columns=None,
    )
):
    print(message)

print("Results plots finished.")
